# 02 — Carbonmark price pull

**Goal:** pull live listing prices from the Carbonmark API for a sample of the project_ids we found in notebook 01, and save them to `data/interim` so notebook 03 can join them to the OffsetsDB project data.

**You need a Carbonmark sandbox API key for this notebook to run.** Get one free at the [Carbonmark Developer Dashboard](https://docs.carbonmark.com/carbonmark-api/quickstart) — sign up, then generate a key from the Keys page. Paste it into the cell below. Never commit this key to a public GitHub repo — it's in `.gitignore` already via the `.env` pattern, but if you paste it directly into this notebook, strip it out again before making the repo public.

**What this notebook does NOT do:** build a historical price time series. Carbonmark's `/prices` endpoint returns live listings — a snapshot of right now. If you want history, you'd need to re-run this notebook on a schedule (e.g. weekly) and append to the saved output rather than overwrite it. That's flagged as a known limitation in the README, not solved here.

### Working directory

Same note as notebook 03: this cell finds the repo root (by locating `requirements.txt`) and changes into it, so `data/interim` paths work consistently regardless of Colab's default working directory.

In [ ]:
import os

def find_and_enter_repo_root(marker_file="requirements.txt", max_up=5):
    current = os.getcwd()
    for _ in range(max_up):
        if os.path.exists(os.path.join(current, marker_file)):
            os.chdir(current)
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    return None

repo_root = find_and_enter_repo_root()
if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repo root (looking for requirements.txt). Make sure the full "
        "carbon-credit-quant/ folder is uploaded to Colab, not just this notebook file."
    )
print(f"Working directory set to repo root: {repo_root}")

In [ ]:
import os

# Prefers the CARBONMARK_API_KEY environment variable if set (this is how the
# scheduled GitHub Actions workflow supplies it, via a repo Secret — see
# .github/workflows/refresh_prices.yml). Falls back to the placeholder below
# for manual/interactive runs in Colab, where pasting it directly here is fine
# for a free sandbox key.
CARBONMARK_API_KEY = os.environ.get("CARBONMARK_API_KEY", "PASTE_YOUR_SANDBOX_KEY_HERE")

if CARBONMARK_API_KEY == "PASTE_YOUR_SANDBOX_KEY_HERE":
    raise ValueError(
        "You need a real Carbonmark sandbox API key before running this notebook. "
        "Get one free at https://docs.carbonmark.com/carbonmark-api/quickstart, then "
        "either paste it above (fine for manual Colab runs) or set it as the "
        "CARBONMARK_API_KEY environment variable (required for the scheduled "
        "GitHub Actions run)."
    )

## Step 1 — Load the project IDs we want prices for

We reuse the checkpoint saved by notebook 01. Carbonmark won't have a price for every OffsetsDB project — most projects aren't listed on Carbonmark's marketplace at all — so this is a best-effort pull, not a guarantee of full coverage.

In [ ]:
import pandas as pd
import os

INTERIM_DIR = "data/interim"
projects_df = pd.read_csv(f"{INTERIM_DIR}/projects_raw_loaded.csv")

print(f"Loaded {len(projects_df):,} projects from notebook 01's checkpoint")
print("Sample project_ids:", projects_df["project_id"].head(5).tolist())

## Step 2 — Pull prices from the Carbonmark API

We use the `/carbonProjects` endpoint, which returns project metadata *and* prices together (confirmed from Carbonmark's own API docs), rather than guessing at an endpoint shape. We paginate through results rather than assuming everything comes back in one call — real APIs almost never do.

This cell includes real error handling: a failed request for one page shouldn't crash the whole pull, it should log the failure and continue.

In [ ]:
import requests
import time
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

BASE_URL = "https://v19.api.carbonmark.com"  # sandbox base URL per Carbonmark's own docs
HEADERS = {"Authorization": f"Bearer {CARBONMARK_API_KEY}"}


def fetch_carbonmark_projects(max_pages: int = 20, page_size: int = 100):
    """
    Pull carbon projects (with pricing) from Carbonmark, paginated.
    Returns a list of raw dicts — cleaning/parsing happens in the next cell,
    kept separate so a parsing bug doesn't mean re-fetching from the API.
    """
    all_results = []
    for page in range(max_pages):
        params = {"limit": page_size, "offset": page * page_size}
        try:
            response = requests.get(f"{BASE_URL}/carbonProjects", headers=HEADERS, params=params, timeout=30)
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            logger.error(f"Request failed on page {page}: {e}")
            break

        data = response.json()
        # Carbonmark's response shape may be a plain list or a dict with a results key —
        # handle both rather than assuming, and log clearly if the shape is unexpected.
        if isinstance(data, list):
            page_results = data
        elif isinstance(data, dict):
            page_results = data.get("items") or data.get("results") or data.get("data") or []
        else:
            logger.warning(f"Unexpected response shape on page {page}: {type(data)}")
            break

        if not page_results:
            logger.info(f"No more results at page {page} — stopping pagination")
            break

        all_results.extend(page_results)
        logger.info(f"Page {page}: got {len(page_results)} projects (total so far: {len(all_results)})")

        time.sleep(0.5)  # be polite to a free sandbox API — don't hammer it

    return all_results


raw_carbonmark_projects = fetch_carbonmark_projects()
print(f"\nTotal projects pulled from Carbonmark: {len(raw_carbonmark_projects)}")
if raw_carbonmark_projects:
    print("\nExample raw record (first result):")
    print(raw_carbonmark_projects[0])

## Step 3 — Parse into a clean prices DataFrame

**Confirmed against the real API response** (not guessed): each project record has a `key` (e.g. `"VCS-844"`), a `price` field (a string, e.g. `"0.1"`), and a `hasSupply` boolean. Critically — **most catalog entries have `price: "0"` and `hasSupply: false`**, meaning no active listing. Those are not zero-price credits, they're not-currently-listed credits, and must be filtered out rather than treated as real price observations.

Also confirmed by actually pulling the full catalog: Carbonmark's `registry` field uses short codes (`VCS`, `UCR`, `ICR`, `CMARK`, `REGEN`, `TVER`, `ECO`, `PUR`, `JCS`, `GS`) that mostly do **not** correspond to OffsetsDB's seven tracked registries. Only `VCS` (Verra) reliably joins. This means real Carbonmark price coverage against OffsetsDB is small and Verra-concentrated — worth knowing before you're surprised by a small final dataset.

In [ ]:
def parse_carbonmark_prices(raw_projects: list) -> pd.DataFrame:
    """
    Extract project_id + price from Carbonmark's raw project records.
    Only keeps records with hasSupply=True and a positive price — everything
    else is a catalog entry with no active listing, not a real price observation.
    Carbonmark's 'key' field (e.g. 'VCS-844') maps to OffsetsDB's project_id
    format by removing the hyphen: 'VCS-844' -> 'VCS844'. Confirmed against
    real OffsetsDB data, not assumed.
    """
    rows = []
    skipped_no_supply = 0
    skipped_bad_price = 0

    for proj in raw_projects:
        key = proj.get("key")
        has_supply = proj.get("hasSupply", False)
        raw_price = proj.get("price")

        if not has_supply:
            skipped_no_supply += 1
            continue

        try:
            price = float(raw_price)
        except (TypeError, ValueError):
            skipped_bad_price += 1
            continue

        if price <= 0 or key is None:
            skipped_bad_price += 1
            continue

        normalized_id = key.replace("-", "")

        rows.append({
            "project_id": normalized_id,
            "carbonmark_key": key,
            "carbonmark_registry": proj.get("registry"),
            "price_usd": price,
            "observed_at": pd.Timestamp.now().isoformat(),
            "source": "carbonmark",
        })

    logger.info(
        f"Parsed {len(rows)} active-listing prices "
        f"(skipped {skipped_no_supply} with no supply, {skipped_bad_price} with invalid price)"
    )
    return pd.DataFrame(rows)


prices_df = parse_carbonmark_prices(raw_carbonmark_projects)
print(f"Parsed {len(prices_df)} active price observations out of {len(raw_carbonmark_projects)} total catalog entries")
prices_df.head(10)

## Step 4 — Check join coverage against OffsetsDB before saving

This is the moment of truth: how many of these Carbonmark prices actually match a project_id we have OffsetsDB characteristics for? If coverage is very low, the ID normalization logic above is probably wrong and needs fixing before we build anything on top of it.

In [ ]:
matched = prices_df["project_id"].isin(projects_df["project_id"])
match_rate = matched.mean() if len(prices_df) > 0 else 0

print(f"Prices with a matching OffsetsDB project_id: {matched.sum()} / {len(prices_df)} ({match_rate:.1%})")

if match_rate < 0.3 and len(prices_df) > 0:
    print("\n⚠️  Match rate is low. Likely causes:")
    print("  - The ID normalization in parse_carbonmark_prices() doesn't match OffsetsDB's actual format")
    print("  - Compare a few unmatched Carbonmark keys directly against projects_df['project_id'].head(20)")
    print("This needs fixing before notebook 03 — a low-coverage join silently shrinks your sample.")

matched_prices_df = prices_df[matched].copy()
print(f"\nProceeding with {len(matched_prices_df)} matched price observations")

## Step 5 — Save checkpoint

Saved with a timestamp in the filename so re-running this notebook later (to build price history) doesn't silently overwrite the previous pull.

In [ ]:
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
output_path = f"{INTERIM_DIR}/carbonmark_prices_{timestamp}.csv"
matched_prices_df.to_csv(output_path, index=False)

# Also save/update a 'latest' pointer file that notebook 03 reads by default,
# so you don't have to update a filename by hand every time you re-run this.
matched_prices_df.to_csv(f"{INTERIM_DIR}/carbonmark_prices_latest.csv", index=False)

print(f"Saved {len(matched_prices_df)} price observations to:")
print(f"  {output_path}")
print(f"  {INTERIM_DIR}/carbonmark_prices_latest.csv (overwritten each run — use this as the 'current' price file)")
print()
print("Re-run this notebook periodically (e.g. weekly) to start building real price history.")
print("Each timestamped file is preserved, so nothing is lost between runs.")